In [14]:
import pandas as pd
from os.path import join, abspath
from bs4 import BeautifulSoup
from bs4.element import NavigableString
import numpy as np
from IPython.display import display

artifacts_abspath = abspath("../../../artifacts")
assets_abspath = abspath("../../../assets")
source_abspath = join(assets_abspath, "evm/opcodes.xlsx")
target_abspath = join(artifacts_abspath, "evm/anki_export.csv")

In [2]:
notes_remove = "(opens in a new tab)"
source = pd.read_excel(source_abspath, sheet_name="Sheet")
source.rename(
    columns={
        "Stack": "Opcode",
        "Name": "Mnemonic",
        "Initial Stack": "Inputs",
        "Resulting Stack": "Output",
    },
    inplace=True,
)
source["Notes"] = source["Notes"].apply(
    lambda v: v.replace(notes_remove, "") if not pd.isna(v) else v
)
source

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes
0,00,STOP,0,NaN,NaN,NaN,NaN,halt execution
1,01,ADD,3,NaN,"a, b",a + b,NaN,(u)int256 addition modulo 2**256
2,02,MUL,5,NaN,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256
3,03,SUB,3,NaN,"a, b",a - b,NaN,(u)int256 addition modulo 2**256
4,04,DIV,5,NaN,"a, b",a // b,NaN,uint256 division
...,...,...,...,...,...,...,...,...
151,FA,STATICCALL,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,"gas, addr, argOst, argLen, retOst, retLen",success,mem[retOst:retOst+retLen-1] := returndata,NaN
152,FB-FC,invalid,NaN,NaN,NaN,NaN,NaN,NaN
153,FD,REVERT,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,"ost, len",.,NaN,revert(mem[ost:ost+len-1])
154,FE,INVALID,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,NaN,NaN,NaN,designated invalid opcode - EIP-141


In [4]:
def create_back(row):
    s = BeautifulSoup("", "html.parser")
    h2 = s.new_tag("h2")
    h2.string = f"{row['Mnemonic']} ({row['Opcode']})"

    dl = s.new_tag("dl", style="text-align: left;")
    dt_styles = "font-weight: bold;"

    if not pd.isna(row["Notes"]):
        notes_dt = s.new_tag("dt", style=dt_styles)
        notes_dt.append(NavigableString("Notes"))
        notes_dd = s.new_tag("dd")
        notes_dd.append(NavigableString(row["Notes"]))
        dl.append(notes_dt)
        dl.append(notes_dd)

    gas_dt = s.new_tag("dt", style=dt_styles)
    gas_dt.append(NavigableString("Gas"))
    gas_dd = s.new_tag("dd")
    if not pd.isna(row["Gas Constant"]):
        gas_dd.append(NavigableString(str(row["Gas Constant"])))
    if not pd.isna(row["Gas Dynamic"]):
        gas_dyn = s.new_tag("a", href=row["Gas Dynamic"])
        gas_dyn.string = "dynamic"
        if not pd.isna(row["Gas Constant"]):
            gas_dd.append(NavigableString(" + "))
        gas_dd.append(gas_dyn)
    if pd.isna(row["Gas Constant"]) and pd.isna(row["Gas Dynamic"]):
        gas_dd.append(NavigableString("0"))
    dl.append(gas_dt)
    dl.append(gas_dd)

    if not pd.isna(row["Inputs"]):
        inputs_dt = s.new_tag("dt", style=dt_styles)
        inputs_dt.append(NavigableString("Inputs"))
        inputs_dd = s.new_tag("dd")
        inputs_dd.append(NavigableString(row["Inputs"]))
        dl.append(inputs_dt)
        dl.append(inputs_dd)

    if not pd.isna(row["Output"]):
        outputs_dt = s.new_tag("dt", style=dt_styles)
        outputs_dt.append(NavigableString("Output"))
        outputs_dd = s.new_tag("dd")
        outputs_dd.append(NavigableString(row["Output"]))
        dl.append(outputs_dt)
        dl.append(outputs_dd)

    if not pd.isna(row["Mem / Storage"]):
        mem_dt = s.new_tag("dt", style=dt_styles)
        mem_dt.append(NavigableString("Mem / Storage"))
        mem_dd = s.new_tag("dd")
        mem_dd.append(NavigableString(row["Mem / Storage"]))
        dl.append(mem_dt)
        dl.append(mem_dd)

    s.append(h2)
    s.append(dl)
    return s.prettify()

In [15]:
df = source.copy()
df["Back"] = df.apply(create_back, axis=1)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df)

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes,Back
0,00,STOP,0,NaN,NaN,NaN,NaN,halt execution,"<h2>\n STOP (00)\n</h2>\n<dl style=""text-align..."
1,01,ADD,3,NaN,"a, b",a + b,NaN,(u)int256 addition modulo 2**256,"<h2>\n ADD (01)\n</h2>\n<dl style=""text-align:..."
2,02,MUL,5,NaN,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256,"<h2>\n MUL (02)\n</h2>\n<dl style=""text-align:..."
3,03,SUB,3,NaN,"a, b",a - b,NaN,(u)int256 addition modulo 2**256,"<h2>\n SUB (03)\n</h2>\n<dl style=""text-align:..."
4,04,DIV,5,NaN,"a, b",a // b,NaN,uint256 division,"<h2>\n DIV (04)\n</h2>\n<dl style=""text-align:..."
5,05,SDIV,5,NaN,"a, b",a // b,NaN,int256 division,"<h2>\n SDIV (05)\n</h2>\n<dl style=""text-align..."
6,06,MOD,5,NaN,"a, b",a % b,NaN,uint256 modulus,"<h2>\n MOD (06)\n</h2>\n<dl style=""text-align:..."
7,07,SMOD,5,NaN,"a, b",a % b,NaN,int256 modulus,"<h2>\n SMOD (07)\n</h2>\n<dl style=""text-align..."
8,08,ADDMOD,8,NaN,"a, b, N",(a + b) % N,NaN,(u)int256 addition modulo N,"<h2>\n ADDMOD (08)\n</h2>\n<dl style=""text-ali..."
9,09,MULMOD,8,NaN,"a, b, N",(a * b) % N,NaN,(u)int256 multiplication modulo N,"<h2>\n MULMOD (09)\n</h2>\n<dl style=""text-ali..."


In [5]:
def create_front(row, title):
    s = BeautifulSoup("", "html.parser")
    kind = s.new_tag("p")
    kind.append(NavigableString(title))
    s.append(kind)
    question = s.new_tag("p")
    question.append(NavigableString(str(row["Front"])))
    s.append(question)
    return s.prettify()

In [6]:
def create_set(column):
    mnemonic = df.copy()
    mnemonic.rename(columns={column: "Front"}, inplace=True)
    mnemonic["Front"] = mnemonic.apply(create_front, title=column, axis=1)
    mnemonic.drop(
        columns=[c for c in mnemonic.columns if c not in ["Back", "Front"]],
        inplace=True,
    )
    mnemonic.drop_duplicates(subset=["Front"], inplace=True)
    mnemonic.dropna(subset=["Front"], inplace=True)
    mnemonic["Tags"] = column
    return mnemonic

In [7]:
concat = pd.concat(
    [
        create_set("Opcode"),
        create_set("Mnemonic"),
        create_set("Output"),
        create_set("Notes"),
    ],
    ignore_index=True,
)
concat

,Front,Back,Tags
0,<p>\n Opcode\n</p>\n<p>\n 00\n</p>\n,"<h2>\n STOP (00)\n</h2>\n<dl style=""text-align...",Opcode
1,<p>\n Opcode\n</p>\n<p>\n 01\n</p>\n,"<h2>\n ADD (01)\n</h2>\n<dl style=""text-align:...",Opcode
2,<p>\n Opcode\n</p>\n<p>\n 02\n</p>\n,"<h2>\n MUL (02)\n</h2>\n<dl style=""text-align:...",Opcode
3,<p>\n Opcode\n</p>\n<p>\n 03\n</p>\n,"<h2>\n SUB (03)\n</h2>\n<dl style=""text-align:...",Opcode
4,<p>\n Opcode\n</p>\n<p>\n 04\n</p>\n,"<h2>\n DIV (04)\n</h2>\n<dl style=""text-align:...",Opcode
...,...,...,...
525,<p>\n Notes\n</p>\n<p>\n return mem[ost:ost+le...,"<h2>\n RETURN (F3)\n</h2>\n<dl style=""text-ali...",Notes
526,<p>\n Notes\n</p>\n<p>\n addr = keccak256(0xff...,"<h2>\n CREATE2 (F5)\n</h2>\n<dl style=""text-al...",Notes
527,<p>\n Notes\n</p>\n<p>\n revert(mem[ost:ost+le...,"<h2>\n REVERT (FD)\n</h2>\n<dl style=""text-ali...",Notes
528,<p>\n Notes\n</p>\n<p>\n designated invalid op...,"<h2>\n INVALID (FE)\n</h2>\n<dl style=""text-al...",Notes


In [8]:
concat.to_csv(
    target_abspath, sep="|", index=False, header=False, encoding="utf-8"
)